In [ ]:
################# IMPACT OF NUMBER OF REUSED STEPS - STRESS=0 #################
#
#
# =========================================================
# Cell 1 – install required libraries (run first)
# =========================================================
!pip install pandas matplotlib numpy



In [1]:
from sma import Config, ReportIO

In [48]:
cnf = Config.from_file("sma-measurments.yaml")
files = {
    "full_reuse_0_stress": ReportIO.load_from_location("data/step_reuse_1000_baremetal_2026_02_04_16_46_47_dc9db2e2",cnf),
    "full_reuse_5_stress": ReportIO.load_from_location("data/step_reuse_1000_baremetal_2026_02_04_16_47_58_3604741e",cnf),
    "full_reuse_50_stress": ReportIO.load_from_location("data/step_reuse_1000_baremetal_2026_02_04_16_49_09_a4d7701b",cnf),
    "4-step_reuse_50_stress": ReportIO.load_from_location("data/step_reuse_1000_baremetal_2026_02_04_16_39_46_58dc56eb",cnf),
    "4-step_reuse_0_stress": ReportIO.load_from_location("data/step_reuse_1000_baremetal_2026_02_04_16_37_15_855de02e",cnf),
}

In [49]:
import pandas as pd

def compare(metric_name, p_name="pod_name"):
    all_agg = None
    for name, report in files.items():
        df = report.get_dataframe(metric_name)
        duration = report.metadata.run.duration().total_seconds()
        df["type"] = name
        df["duration"] = duration
        if all_agg is None:
            all_agg = df
        else:
            all_agg = pd.concat([all_agg, df], ignore_index=True)
    all_agg["pod"] = all_agg[p_name].apply(lambda x: "-".join(x.split("-")[0:2]))
    durations = all_agg.groupby(["type"])["duration"].first().reset_index()
    return all_agg.groupby(["type", "pod"])[metric_name].agg(lambda x: x.max() - x.min()), durations
    

In [50]:
compare("kepler_pod_cpu_power_joules_raw")

(type                    pod                
 4-step_reuse_0_stress   data-preprocess        24.906996
                         data-quality           22.223346
                         generate-address       17.614362
                         generate-birthday      23.215308
                         generate-id            23.478769
                         generate-name          18.449088
                         load-data              17.540248
                         parallel-full           0.096161
                         privacy-tracking       31.922441
                         synthetic-generator    27.667038
 4-step_reuse_50_stress  data-preprocess        24.905014
                         data-quality           22.249308
                         generate-address       17.624803
                         generate-birthday      23.237337
                         generate-id            23.494995
                         generate-name          18.471286
                         lo

In [36]:
compare("system_cpu_usage_raw","pod")

type                    pod                
4-step_reuse_50_stress  data-preprocess        12.922583
                        data-quality           13.467631
                        generate-address        9.791814
                        generate-birthday       9.696988
                        generate-id            10.104486
                        generate-name          10.165627
                        load-data               9.429538
                        parallel-full           0.071237
                        privacy-tracking       13.276230
                        synthetic-generator    12.312170
full_reuse_0_stress     data-preprocess        12.649790
                        data-quality           13.195677
                        generate-address        9.550496
                        generate-birthday       9.443932
                        generate-id             9.856297
                        generate-name           9.931651
                        load-data           